# F1 Telemetry Analysis 🏎️

Comprehensive analysis of Formula 1 telemetry data including driver performance, speed analysis, and lap comparisons.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Set style for better visualizations
sns.set_style('darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)
print('✅ Libraries imported successfully!')

## 2. Load Telemetry Data

In [ ]:
# Sample F1 telemetry data
# Replace this with your actual data source

# Create sample telemetry data for demonstration
np.random.seed(42)
laps = 50

telemetry_data = {
    'Lap': np.arange(1, laps + 1),
    'Driver': np.random.choice(['Verstappen', 'Hamilton', 'Leclerc'], laps),
    'Speed_kmh': np.random.uniform(280, 340, laps),
    'Throttle_%': np.random.uniform(50, 100, laps),
    'Brake_%': np.random.uniform(0, 100, laps),
    'Fuel_kg': np.linspace(100, 10, laps),
    'Tire_Temp_C': np.random.uniform(80, 110, laps),
    'DRS_Active': np.random.choice([0, 1], laps, p=[0.7, 0.3]),
    'Lap_Time_s': np.random.uniform(85, 95, laps)
}

df = pd.DataFrame(telemetry_data)

print('📊 Telemetry Data Loaded!')
print(f'Shape: {df.shape}')
print('\nFirst 5 rows:')
print(df.head())

## 3. Data Exploration & Summary

In [ ]:
# Data Info
print('📋 Data Information:')
print(df.info())
print('\n')

# Statistical Summary
print('📈 Statistical Summary:')
print(df.describe())
print('\n')

# Missing Values
print('❌ Missing Values:')
print(df.isnull().sum())

## 4. Driver Performance Analysis

In [ ]:
# Driver statistics
driver_stats = df.groupby('Driver').agg({
    'Speed_kmh': ['mean', 'max', 'min'],
    'Lap_Time_s': ['mean', 'min'],
    'Throttle_%': 'mean',
    'Brake_%': 'mean',
    'DRS_Active': 'sum'
}).round(2)

print('🏁 Driver Performance Stats:')
print(driver_stats)
print('\n')

# Best lap per driver
best_laps = df.loc[df.groupby('Driver')['Lap_Time_s'].idxmin()]
print('⚡ Best Lap Times by Driver:')
print(best_laps[['Driver', 'Lap', 'Lap_Time_s', 'Speed_kmh']].sort_values('Lap_Time_s'))

## 5. Speed Analysis & Visualization

In [ ]:
# Speed over laps
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Speed progression
for driver in df['Driver'].unique():
    driver_data = df[df['Driver'] == driver]
    axes[0, 0].plot(driver_data['Lap'], driver_data['Speed_kmh'], marker='o', label=driver, alpha=0.7)

axes[0, 0].set_xlabel('Lap Number')
axes[0, 0].set_ylabel('Speed (km/h)')
axes[0, 0].set_title('Speed Progression Over Laps')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Plot 2: Average speed by driver
avg_speed = df.groupby('Driver')['Speed_kmh'].mean().sort_values(ascending=False)
axes[0, 1].bar(avg_speed.index, avg_speed.values, color=['red', 'blue', 'yellow'])
axes[0, 1].set_ylabel('Average Speed (km/h)')
axes[0, 1].set_title('Average Speed by Driver')
axes[0, 1].grid(True, alpha=0.3, axis='y')

# Plot 3: Lap times
df.boxplot(column='Lap_Time_s', by='Driver', ax=axes[1, 0])
axes[1, 0].set_xlabel('Driver')
axes[1, 0].set_ylabel('Lap Time (seconds)')
axes[1, 0].set_title('Lap Time Distribution by Driver')
plt.sca(axes[1, 0])
plt.xticks(rotation=0)

# Plot 4: Throttle vs Brake correlation
axes[1, 1].scatter(df['Throttle_%'], df['Brake_%'], alpha=0.6, s=50)
axes[1, 1].set_xlabel('Throttle (%)')
axes[1, 1].set_ylabel('Brake (%)')
axes[1, 1].set_title('Throttle vs Brake Usage')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('speed_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

print('📊 Visualization saved as speed_comparison.png')

## 6. Fuel & Tire Analysis

In [ ]:
# Fuel consumption analysis
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Fuel over laps
axes[0].plot(df['Lap'], df['Fuel_kg'], marker='o', color='green', linewidth=2)
axes[0].fill_between(df['Lap'], df['Fuel_kg'], alpha=0.3, color='green')
axes[0].set_xlabel('Lap Number')
axes[0].set_ylabel('Fuel Remaining (kg)')
axes[0].set_title('Fuel Consumption Over Race')
axes[0].grid(True, alpha=0.3)

# Tire temperature
axes[1].plot(df['Lap'], df['Tire_Temp_C'], marker='s', color='orange', linewidth=2)
axes[1].axhline(y=df['Tire_Temp_C'].mean(), color='red', linestyle='--', label='Average Temp')
axes[1].set_xlabel('Lap Number')
axes[1].set_ylabel('Tire Temperature (°C)')
axes[1].set_title('Tire Temperature Progression')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'⛽ Average Fuel Consumption per Lap: {(df["Fuel_kg"].iloc[0] - df["Fuel_kg"].iloc[-1]) / len(df):.2f} kg')
print(f'🌡️ Average Tire Temperature: {df["Tire_Temp_C"].mean():.2f}°C')
print(f'🔥 Max Tire Temperature: {df["Tire_Temp_C"].max():.2f}°C')

## 7. DRS (Drag Reduction System) Analysis

In [ ]:
# DRS Impact Analysis
drs_active = df[df['DRS_Active'] == 1]
drs_inactive = df[df['DRS_Active'] == 0]

print('🚀 DRS Impact Analysis:')
print(f'\nDRS Active:')
print(f'  - Average Speed: {drs_active["Speed_kmh"].mean():.2f} km/h')
print(f'  - Average Throttle: {drs_active["Throttle_%"].mean():.2f}%')
print(f'  - Frequency: {len(drs_active)} laps')

print(f'\nDRS Inactive:')
print(f'  - Average Speed: {drs_inactive["Speed_kmh"].mean():.2f} km/h')
print(f'  - Average Throttle: {drs_inactive["Throttle_%"].mean():.2f}%')
print(f'  - Frequency: {len(drs_inactive)} laps')

speed_gain = drs_active['Speed_kmh'].mean() - drs_inactive['Speed_kmh'].mean()
print(f'\n⚡ Speed Gain with DRS: {speed_gain:.2f} km/h')

# Visualization
fig, ax = plt.subplots(figsize=(10, 6))
drs_comparison = pd.DataFrame({
    'DRS OFF': [drs_inactive['Speed_kmh'].mean(), drs_inactive['Throttle_%'].mean()],
    'DRS ON': [drs_active['Speed_kmh'].mean(), drs_active['Throttle_%'].mean()]
}, index=['Average Speed (km/h)', 'Average Throttle (%)'])

drs_comparison.plot(kind='bar', ax=ax, color=['#FF1E3E', '#00D26E'], width=0.7)
ax.set_ylabel('Value')
ax.set_title('DRS System Impact on Performance')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## 8. Key Insights & Summary

In [ ]:
print('='*60)
print('🏁 F1 TELEMETRY ANALYSIS - KEY INSIGHTS 🏁')
print('='*60)

print('\n📊 OVERALL STATISTICS:')
print(f'  Total Laps Analyzed: {len(df)}')
print(f'  Number of Drivers: {df["Driver"].nunique()}')
print(f'  Average Race Speed: {df["Speed_kmh"].mean():.2f} km/h')
print(f'  Fastest Speed Recorded: {df["Speed_kmh"].max():.2f} km/h')
print(f'  Slowest Speed Recorded: {df["Speed_kmh"].min():.2f} km/h')

print('\n🏆 TOP PERFORMER:')
fastest_driver = df.loc[df['Lap_Time_s'].idxmin()]
print(f'  Driver: {fastest_driver["Driver"]}')
print(f'  Best Lap Time: {fastest_driver["Lap_Time_s"]:.2f}s')
print(f'  Top Speed on Best Lap: {fastest_driver["Speed_kmh"]:.2f} km/h')

print('\n⚡ PERFORMANCE HIGHLIGHTS:')
print(f'  Average Throttle Usage: {df["Throttle_%"].mean():.2f}%')
print(f'  Average Brake Application: {df["Brake_%"].mean():.2f}%')
print(f'  DRS Usage Rate: {(df["DRS_Active"].sum() / len(df) * 100):.1f}%')
print(f'  Average Tire Temperature: {df["Tire_Temp_C"].mean():.2f}°C')

print('\n🔧 RECOMMENDATIONS:')
print('  1. Monitor tire temperatures during high-speed sections')
print('  2. Optimize DRS usage for straights to maximize speed')
print('  3. Analyze brake patterns to improve cornering efficiency')
print('  4. Manage fuel consumption for race strategy')

print('\n' + '='*60)
print('✅ Analysis Complete!')
print('='*60)

## 9. Export Results

In [ ]:
# Export data to CSV
df.to_csv('f1_telemetry_analysis.csv', index=False)
print('✅ Telemetry data exported to f1_telemetry_analysis.csv')

# Export driver stats
driver_stats.to_csv('driver_performance_stats.csv')
print('✅ Driver statistics exported to driver_performance_stats.csv')

# Summary report
summary = {
    'Total Laps': len(df),
    'Number of Drivers': df['Driver'].nunique(),
    'Average Speed (km/h)': round(df['Speed_kmh'].mean(), 2),
    'Best Lap Time (s)': round(df['Lap_Time_s'].min(), 2),
    'Average Fuel Consumption': round((df['Fuel_kg'].iloc[0] - df['Fuel_kg'].iloc[-1]) / len(df), 2)
}

print('\n📋 Analysis Summary:')
for key, value in summary.items():
    print(f'  {key}: {value}')